# AA4 — Performance Analysis

> **Recommended:** run `python scripts/perf_charts.py` from the project root instead.
> That script produces the same charts without needing a Jupyter kernel.

This notebook reads `reports/perf_metrics.jsonl` (written by `utils/metrics.measure()` during each pipeline run) and produces the before/after charts for the three experiments documented in `reports/perf_report.md`.

**Run the pipeline at least once before opening this notebook:**
```bash
make pipeline          # Docker
# or
.\run_local.ps1 -Stage pipeline   # Windows local
```

In [ ]:
import json
import pathlib
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

METRICS_FILE = pathlib.Path("../reports/perf_metrics.jsonl")

records = []
with METRICS_FILE.open() as f:
    for line in f:
        line = line.strip()
        if line:
            records.append(json.loads(line))

df = pd.DataFrame(records)
print(f"Loaded {len(df)} records from {METRICS_FILE}")
df.head(10)

## Overview — All stages, all runs

In [ ]:
pivot = (
    df.groupby(["label", "aqe"])["wall_sec"]
    .mean()
    .unstack("aqe")
    .rename(columns={True: "AQE ON", False: "AQE OFF"})
    .sort_values("AQE ON", ascending=False)
)
pivot

## Experiment 1 — AQE On vs Off

Stage: `silver.charts_enriched` (the large charts × tracks sort-merge join). AQE dynamically coalesces post-shuffle partitions, reducing task count in the join stage.

In [ ]:
stage = "silver.charts_enriched"
exp1 = df[df["label"] == stage].groupby("aqe")["wall_sec"].mean()

fig, ax = plt.subplots(figsize=(6, 4))
labels = ["AQE OFF", "AQE ON"]
values = [exp1.get(False, 0), exp1.get(True, 0)]
colors = ["#e74c3c", "#2ecc71"]
bars = ax.bar(labels, values, color=colors, width=0.5, edgecolor="white")
ax.bar_label(bars, fmt="%.1fs", padding=4, fontsize=11, fontweight="bold")
ax.set_ylabel("Wall time (s)")
ax.set_title(f"Experiment 1 — AQE impact on '{stage}'")
ax.set_ylim(0, max(values) * 1.25 if values else 1)
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.0fs'))
ax.spines[["top", "right"]].set_visible(False)

if all(v > 0 for v in values):
    delta_pct = (values[0] - values[1]) / values[0] * 100
    ax.annotate(
        f"↓ {delta_pct:.0f}% improvement",
        xy=(1, values[1]), xytext=(1.2, (values[0] + values[1]) / 2),
        arrowprops=dict(arrowstyle="->", color="gray"),
        fontsize=10, color="gray",
    )

plt.tight_layout()
plt.savefig("../reports/spark_ui/exp1_aqe_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"AQE OFF: {values[0]:.1f}s   AQE ON: {values[1]:.1f}s   Δ = {values[0]-values[1]:.1f}s ({(values[0]-values[1])/max(values[0],0.001)*100:.0f}%)")

## Experiment 2 — Broadcast hint vs default

The countries table has 63 rows (~5 KB). The explicit `broadcast(countries)` in `silver/join_enriched.py` eliminates the shuffle for the right side. We compare `silver.countries` join wall time with/without the hint.

In [ ]:
# Filter to silver.charts_enriched (where the broadcast join happens)
exp2 = df[df["label"] == "silver.charts_enriched"].groupby("aqe")["wall_sec"].agg(["mean", "min", "max"])
print("silver.charts_enriched timings (broadcast join is always active in current code):")
print(exp2.to_string())
print()
print("Note: to get a before/after for broadcast, temporarily remove broadcast() in")
print("src/bigdata_music/silver/join_enriched.py:57 and re-run Silver only.")

## Experiment 3 — Pre-repartition before sort-merge join

In [ ]:
# Full pipeline wall time comparison: AQE ON vs OFF
total = df.groupby("aqe")["wall_sec"].sum().reset_index()
total["label"] = total["aqe"].map({True: "AQE ON\n(+ repartition)", False: "AQE OFF\n(no repartition)"})

fig, ax = plt.subplots(figsize=(7, 4))
colors = ["#e74c3c" if not v else "#2ecc71" for v in total["aqe"]]
bars = ax.bar(total["label"], total["wall_sec"], color=colors, width=0.5, edgecolor="white")
ax.bar_label(bars, fmt="%.0fs", padding=4, fontsize=11, fontweight="bold")
ax.set_ylabel("Total wall time (s)")
ax.set_title("Experiment 3 — Total pipeline time (all stages)")
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.0fs'))
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.savefig("../reports/spark_ui/exp3_total_pipeline.png", dpi=150, bbox_inches="tight")
plt.show()

## All-stages breakdown

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

stages = df["label"].unique()
x = range(len(stages))
width = 0.35

aqe_on  = [df[(df["label"] == s) & (df["aqe"] == True)]["wall_sec"].mean()  for s in stages]
aqe_off = [df[(df["label"] == s) & (df["aqe"] == False)]["wall_sec"].mean() for s in stages]

import numpy as np
aqe_on  = [v if not np.isnan(v) else 0 for v in aqe_on]
aqe_off = [v if not np.isnan(v) else 0 for v in aqe_off]

b1 = ax.bar([i - width/2 for i in x], aqe_off, width, label="AQE OFF", color="#e74c3c", alpha=0.85)
b2 = ax.bar([i + width/2 for i in x], aqe_on,  width, label="AQE ON",  color="#2ecc71", alpha=0.85)

ax.set_xticks(list(x))
ax.set_xticklabels(stages, rotation=30, ha="right", fontsize=9)
ax.set_ylabel("Wall time (s)")
ax.set_title("Per-stage wall time: AQE ON vs OFF")
ax.legend()
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.savefig("../reports/spark_ui/all_stages_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

## Summary table for perf_report.md

In [ ]:
summary = (
    df.groupby(["label", "aqe"])["wall_sec"]
    .agg(["mean", "min", "max", "count"])
    .round(1)
)
print(summary.to_markdown())